In [2]:
# We only need 3 packages that are readily available
import requests
import pandas as pd
import numpy as np

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [4]:
# Welcome to my projection tool! Please set some variables we'll need later:
# League ID actually seems to change each year. This has to be updated to get the latest rosters
league_id = '1349152931005333504'

# Update the year to take latest updates
year = 2026

# Value for the next week to be played. Weeks before this take their recorded values, after is simmed
next_wk = 3

In [5]:
# All the real NFL players that exist. This includes their team (relevant for bye) and fantasy position
every_player = requests.get('https://api.sleeper.app/v1/players/nfl').json()

In [27]:
#structure of the data
print(len(every_player.keys())) # there are 12229 players in the directory
print(len(every_player.values())) # each player is associated with 53 different variables

12229
12229


In [30]:
player_ids = list(every_player.keys()) 
#print(player_ids)

for i in range(0,len(player_ids)):
    print(f"name:{every_player[player_ids[i]]['full_name']}, age:{every_player[player_ids[i]]['age']}, position:{every_player[player_ids[i]]['position']}, team:{every_player[player_ids[i]]['team']}, team_changed_at: {every_player[player_ids[i]]['team_changed_at']}, years_exp: {every_player[player_ids[i]]['years_exp']}")


#print(len(every_player['6462']))
#print(every_player['6462'])

#what do i need from each player: name, position, team, points per week, (active each week?)

6462
name:Moliki Matavao, age:23, position:TE, team:NO, team_changed_at: None, years_exp: 1
11255
name:Mike Maccagnan, age:None, position:None, team:None, team_changed_at: None, years_exp: 0
8842
name:Jake Briningstool, age:23, position:TE, team:KC, team_changed_at: None, years_exp: 1
13940


IndexError: list index out of range

In [6]:
# A series of blank lists. These will be populated and then combined into a Data Frame
pls = []
poss=[]
wk1=[]
wk2=[]
wk3=[]
wk4=[]
wk5=[]
wk6=[]
wk7=[]
wk8=[]
wk9=[]
wk10=[]
wk11=[]
wk12=[]
wk13=[]
wk14=[]
wk15=[]
wk16=[]
wk17=[]
wk18=[]

In [7]:
# Iterate through each player and get their projections. There's a counter here as this has sometimes got stuck
my_l = 0
for team in rosters:
    for player in team.get('players'):
        # Each player on each team
        print(my_l)
        my_l+=1
        # Get their position and team from the data we pulled earlier and look up their bye
        my_pos = every_player.get(player).get('position')
        # For each player (capped at 170 for our league), make an API call that gives all of their projections for each week
        my_p = requests.get('https://api.sleeper.com/projections/nfl/player/'+player+'?season='+str(year)+'&season_type=regular&grouping=week').json()
        # Start to write to our lists
        pls+=[player]
        poss+=[my_pos]
        # For any weeks in the past, just set to 0. We need a value but we don't use it
        for week in range(1,next_wk):
            locals()[f'wk{week}']+=[0]
        # Point projections vary by position. Moved above the week iteration for efficiency
        if my_pos == 'QB':
            for week in range(next_wk,19):
                # We use a try/except here as if a player is on bye or injured, they do not have any values and would just throw an error so give them a 0 in these instances
                # noinspection PyBroadException
                try:
                    # Full stat calculations taken from our league rules
                    w_st = dict(my_p.get(str(week)).get('stats'))
                    locals()[f'wk{week}']+=[w_st.get('pass_yd',0) * 0.04 + w_st.get('pass_td',0) * 5 + w_st.get('pass_2pt',0) * 2 - w_st.get('pass_int',0) * 2 - w_st.get('pass_sack',0) * 0.5 +  w_st.get('rush_yd',0) * 0.1 + w_st.get('rush_td',0) * 6 + w_st.get('rush_fd',0) * 0.5 + w_st.get('rush_2pt',0) * 2 - w_st.get('fum_lost',0) * 2]
                except:
                    locals()[f'wk{week}']+=[0]
        elif my_pos in ['RB','WR','TE']:
            for week in range(next_wk,19):
                # noinspection PyBroadException
                try:
                    w_st = dict(my_p.get(str(week)).get('stats'))
                    locals()[f'wk{week}']+=[w_st.get('rec_yd',0) * 0.1 + w_st.get('rec_td',0) * 6 + w_st.get('rec_2pt',0) * 2 + w_st.get('rec_fd',0) * 0.5 + float(w_st.get('rec')) * 0.5 +  w_st.get('rush_yd',0) * 0.1 + w_st.get('rush_td',0) * 6 + w_st.get('rush_fd',0) * 0.5 + w_st.get('rush_2pt',0) * 2 - w_st.get('fum_lost',0) * 2]
                except:
                    locals()[f'wk{week}']+=[0]
        elif my_pos == 'K':
            for week in range(next_wk,19):
                # noinspection PyBroadException
                try:
                    # There's a -0.03 to account for missed field goals of 30-39 yards
                    w_st = dict(my_p.get(str(week)).get('stats'))
                    locals()[f'wk{week}']+=[w_st.get('fgm',0) * 3 + w_st.get('fgm_40_49',0) * 1 + w_st.get('fgm_50p',0) * 2 + w_st.get('xpm',0) * 1 - w_st.get('xpmiss',0) * 2 - (w_st.get('fga',0) - w_st.get('fgm',0) - w_st.get('fgmiss_40_49',0) - w_st.get('fgmiss_50p',0)) - 0.03]
                except:
                    locals()[f'wk{week}']+=[0]
        elif my_pos =='DEF':
            for week in range(next_wk,19):
                try:
                    # Bunch of buckets for points and yds allowed scores. We also have +2.68 to account for safeties, 3 and outs, 4th down stops, ST TDs and ST forced fumbles. Through 5 weeks, this was worth about 2.68 pts
                    w_st = dict(my_p.get(str(week)).get('stats'))
                    tot=0
                    if w_st.get('yds_allow',0) < 100:
                        tot+=5
                    elif w_st.get('yds_allow',0) < 200:
                        tot+=3
                    elif w_st.get('yds_allow',0) < 300:
                        tot+=1
                    elif w_st.get('yds_allow',0) >= 500:
                        tot-=2
                    elif w_st.get('yds_allow',0) >= 450:
                        tot-=1

                    if w_st.get('pts_allow',0) < 1:
                        tot+=10
                    elif w_st.get('pts_allow',0) < 7:
                        tot+=7
                    elif w_st.get('pts_allow',0) < 14:
                        tot+=4
                    elif w_st.get('pts_allow',0) < 21:
                        tot+=1
                    elif w_st.get('pts_allow',0) >= 35:
                        tot-=4
                    elif w_st.get('pts_allow',0) >= 28:
                        tot-=1
                    locals()[f'wk{week}'] += [tot + w_st.get('def_td',0) * 6 + w_st.get('sack',0) * 1 + w_st.get('int',0) * 2 + w_st.get('fum_rec',0) * 2 + w_st.get('ff',0) * 1 + w_st.get('blk_kick',0) * 2 + 2.68]
                except:
                    locals()[f'wk{week}']+=[0]

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164


In [8]:
# Glue all the lists above into a single df
player_data = pd.DataFrame(
    data={'player_id':pls,
     'player_pos':poss,
     'week1':wk1,
     'week2':wk2,
     'week3':wk3,
     'week4':wk4,
     'week5':wk5,
     'week6':wk6,
     'week7':wk7,
     'week8':wk8,
     'week9':wk9,
     'week10':wk10,
     'week11':wk11,
     'week12':wk12,
     'week13':wk13,
     'week14':wk14,
     'week15':wk15,
     'week16':wk16,
     'week17':wk17},
    index=pls
)

In [9]:
# Reset the lists, we'll need them again in a minute
user_ids=[]
disp_names =[]
wk1=[]
wk2=[]
wk3=[]
wk4=[]
wk5=[]
wk6=[]
wk7=[]
wk8=[]
wk9=[]
wk10=[]
wk11=[]
wk12=[]
wk13=[]
wk14=[]
wk15=[]
wk16=[]
wk17=[]
wk18=[]

In [10]:
# Generate a weekly sim total per user (subs where necessary)
for user_j in users:
    # Add data to lists
    user_ids+=[user_j.get('user_id')]
    disp_names+=[user_j.get('display_name')]
    user=user_j.get('user_id')
    # As before, set points to zero for the past
    for week in range(1,next_wk):
        locals()[f'wk{week}']+=[0]
    for week in range(next_wk,18):
        # We make a dictionary to fill by position
        wk_dict = {}
        for p in player_list.get(user):
            my_position = player_data.loc[p]['player_pos']
            my_score = player_data.loc[p][f'week{week}']
            # noinspection PyPackages
            if my_position in wk_dict.keys():
                wk_dict[my_position]+=[my_score]
            else:
                wk_dict[my_position]=[my_score]
        wk_tot = 0
        # Pick the best QB, if we don't have any not on bye, just project 20 (unlikely to happen as most players have 2 QBs)
        if max(wk_dict['QB'])==0:
            wk_tot+=20
        else:
            wk_tot+= max(wk_dict['QB'])
        # Pick our top 2 RBs and top 2 WRs
        wk_tot+=max(wk_dict['RB'])
        wk_dict['RB'].remove(max(wk_dict['RB']))
        wk_tot+=max(wk_dict['RB'])
        wk_dict['RB'].remove(max(wk_dict['RB']))
        wk_tot+=max(wk_dict['RB'])
        wk_dict['WR'].remove(max(wk_dict['WR']))
        wk_tot+=max(wk_dict['WR'])
        wk_dict['WR'].remove(max(wk_dict['WR']))
        wk_tot+=max(wk_dict['WR'])
        wk_dict['WR'].remove(max(wk_dict['WR']))
        # Pick the best TE or give 8.5 for a waiver pickup
        if max(wk_dict['TE'])==0:
            wk_tot+=8.5
        else:
            wk_tot+= max(wk_dict['TE'])
        # Pick next best WR/RB/TE as flex
        wk_tot+=max(max(wk_dict['RB']),max(wk_dict['WR']),max(wk_dict['TE']))
        # Pick Kicker and defense giving default values if missing
        if 'K' not in wk_dict.keys():
            wk_tot+=8
        elif max(wk_dict['K'])==0:
            wk_tot+=8
        else:
            wk_tot+= max(wk_dict['K'])
        if max(wk_dict['DEF'])==0:
            wk_tot+=9
        else:
            wk_tot+= max(wk_dict['DEF'])
        # Take the total and make that our weekly projection
        locals()[f'wk{week}']+=[wk_tot]


In [11]:
# Put all this into a df
user_data = pd.DataFrame(
    data={'user_id':user_ids,
     'display_name':disp_names,
     'week1':wk1,
     'week2':wk2,
     'week3':wk3,
     'week4':wk4,
     'week5':wk5,
     'week6':wk6,
     'week7':wk7,
     'week8':wk8,
     'week9':wk9,
     'week10':wk10,
     'week11':wk11,
     'week12':wk12,
     'week13':wk13,
     'week14':wk14,
     'week15':wk15,
     'week16':wk16,
     'week17':wk17},
    index=user_ids
)

In [12]:
# Create the matchups for each week - that has already happened
# These matrices track who has played who and what wins they have. This will be used for our total wins and also for H2H tiebreakers
fixt = pd.DataFrame(0,index = user_ids, columns = user_ids)
wins = pd.DataFrame(0,index = user_ids, columns = user_ids)

# For some reason matchups don't use the user ID but instead take the sequence number for each team when sorted numerically. Use a dictionary to lookup where this conversion is needed
lookup = dict(zip(range(1,11),['1113914763710410752','1079160379282067456','871149694171136000','1113918922971045888','1113920165458079744','1114137071750352896','1115358407864848384','1125085303062097920','1125480183223373824','1133744342155378688']))

# All of our points scored to date will be in a dict too
pf_curr = dict(zip(user_ids,[0,0,0,0,0,0,0,0,0,0]))

#Model the season to date, this won't need to be repeated each sim
for reg_wk in range(1,next_wk):
    # Get all the matchups for the week
    my_wk = requests.get('https://api.sleeper.app/v1/league/'+league_id+'/matchups/'+ str(reg_wk)).json()
    # We'll put each week's user IDs here
    my_match = [[],[],[],[],[]]
    # And each week's point totals here
    my_pts = [[],[],[],[],[]]
    for t in my_wk:
        # For each team in the week, place them in the right matchup and put their points there too
        my_match[t['matchup_id']-1]+=[t['roster_id']]
        my_pts[t['matchup_id']-1]+=[t['points']]
        pf_curr[lookup[t['roster_id']]]+=t['points']
    for m in range(5):
        # Go through each matchup and add the fixtures to games played (we transpose as well)
        fixt.loc[lookup[my_match[m][0]],lookup[my_match[m][1]]] += 1
        fixt.loc[lookup[my_match[m][1]],lookup[my_match[m][0]]] += 1
        # Find out who won and award them the win in the win matrix (I don't do ties)
        if my_pts[m][0]>=my_pts[m][1]:
            wins.loc[lookup[my_match[m][0]],lookup[my_match[m][1]]] += 1
        else:
            wins.loc[lookup[my_match[m][1]],lookup[my_match[m][0]]] += 1

In [13]:
# Create the matchups for the games that will need to be simmed
sim_ms = []
sim_pts = []

for week in range(next_wk, 15):
    # Season ends after 14 weeks so sim from the next week up until that point
    my_wk = requests.get('https://api.sleeper.app/v1/league/1182764216653266944/matchups/'+ str(week)).json()
    my_match = [[],[],[],[],[]]
    my_pts = [[],[],[],[],[]]
    for t in my_wk:
        my_match[t['matchup_id']-1]+=[t['roster_id']]
        # Pull out the projected points for each user from the df we created above
        my_pts[t['matchup_id']-1]+=[user_data.loc[lookup[t['roster_id']],f'week{week}']]
    for m in range(5):
        # Add all the fixtures, these won't change per sim so save ourselves some time
        fixt.loc[lookup[my_match[m][0]],lookup[my_match[m][1]]] += 1
        fixt.loc[lookup[my_match[m][1]],lookup[my_match[m][0]]] += 1
    # Add the matchups and projected scores to lists per week. We use these in the sim
    sim_ms+=[my_match]
    sim_pts+=[my_pts]




In [14]:
# Create a function to rank teams at the end of a season
def reg_rank(s_wins,s_fixt,s_pts):
    # Dictionary of all the values of wins we had
    win_dict = {}
    # List with everyone ordered to be populated
    output = []
    for p1 in s_wins.index.to_list():
        # Find the total number of wins per player and add it to the dictionary
        if sum(s_wins.loc[p1]) in win_dict.keys():
            win_dict[sum(s_wins.loc[p1])]+=[p1]
        else:
            win_dict[sum(s_wins.loc[p1])]=[p1]
    # w is a list of all the counts of wins we have. Sort to have most wins first
    w = list(win_dict.keys())
    w.sort(reverse=True)
    for w2 in w:
        # For each valid number of wins, if only one person had that many, add them to the output, otherwise, we enter tiebreak mode
        if len(win_dict[w2])==1:
            output+=win_dict[w2]
        else:
            # SCS is our dictionary of scores for players with that many wins
            scs = {}
            for pl in win_dict[w2]:
                # Num  is number of wins in relevant matchups, denom is number of games played
                num = 0
                denom = 0
                for p2 in win_dict[w2]:
                    num += s_wins.loc[pl,p2]
                    denom += s_fixt.loc[pl,p2]
                # Score is given by this fraction plus a small amount of points scored as an additional tiebreaker
                scs[float(num)/float(denom) + s_pts[pl]/1000000]=pl
            # Get the full points scored and sort again
            s = list(scs.keys())
            s.sort(reverse=True)
            # Iterate through this list and pull out the order of teams tied for wins
            for sc in s:
                output+=[scs[sc]]
    # Now we need to sort out 6th specifically. Top 5 remain the same, for the others, find who had the most points and put them in the list. Then add everyone else in the order they were previously
    final_output = output[:5]
    max_p = 0.0
    best=''
    for o in output[5:]:
        if s_pts[o]>max_p:
            max_p=s_pts[o]
            best=o
    final_output+=[best]
    output.remove(best)
    final_output+=output[5:]
    # Return a list of 10 users in the order they finished the season based on the win and fixture matrix and their points scored
    return final_output



In [15]:
# Sim each season
# Matrix of places 1-10 and players who finished in these spots
res_mat = pd.DataFrame(0,index = user_ids, columns = list(range(1,11)))

for i in range(sim_ct):
    # Little print statement for me to track progress
    if i % 1000 == 0:
        print(i)
    # Start by initialising the matrix for wins and the points dict. We copy over the existing one so we can reinitialise easily every time
    sim_wins = wins.copy()
    sim_pf = pf_curr.copy()
    # Iterate through weeks
    for sim_wk in range(next_wk,15):
        # Get our matchup and score projection to sim
        matchup_pts = sim_pts[sim_wk - next_wk]
        matchup_ms = sim_ms[sim_wk - next_wk]

    # Create 2 values based on the expected scores. Model can be changed, but it's currently based on being normally distributed with the mean as our projection and s.d. as 31% of that. I'd like to refine this. Add the simmed score to our total scores
        for j in range(5):
            mt = matchup_pts[j]
            mu = matchup_ms[j]
            a = np.random.normal(mt[0],0.31 * mt[0])
            b = np.random.normal(mt[1],0.31 * mt[1])
            sim_pf[lookup[mu[0]]]+=a
            sim_pf[lookup[mu[1]]]+=b

    # Compare and make someone the winner
            if a>b:
                sim_wins.loc[lookup[mu[0]],lookup[mu[1]]]+=1
            else:
                sim_wins.loc[lookup[mu[1]],lookup[mu[0]]]+=1

    # Rank the teams using the function above
    rk = reg_rank(sim_wins,fixt,sim_pf)

    # Put these ranks into our res_mat
    for k in range(10):
        res_mat.loc[rk[k],k+1]+=1

# Create a pct res_mat by taking the counts above and dividing by the number of sims. Also add names for clarity
pct_res_mat = pd.DataFrame(None,index = user_ids, columns = ['Display_Name'] + list(range(1,11)))
for u in user_ids:
    pct_res_mat.loc[u,'Display_Name'] = user_data.loc[u,'display_name']
    for x in range(1,11):
        pct_res_mat.loc[u,x]=res_mat.loc[u,x]/sim_ct

# Add another col for playoff chances and bye chances (just adding the percentages for relevant finishes)
pct_res_mat['Playoffs'] = pct_res_mat[2] + pct_res_mat[3] + pct_res_mat[4] + pct_res_mat[5] + pct_res_mat[6] + pct_res_mat[1]

pct_res_mat['Bye'] = pct_res_mat[2] + pct_res_mat[1]

# pct_res_mat is our final output and is what I like to share with the league

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000


In [16]:
pct_res_mat.to_csv('C:/Users/adam.davitt/OneDrive - Accenture/Downloads/output_wk' + str(next_wk-1) + '.csv')